# Spring Valley Search Data Quality Audit

## tl;dr

- **The Search evidence is usable for PDP enrichment planning.** 803 of 850 tasks succeeded (94.5%), all 20,314 successful-result rows passed raw artifact checksum validation, and 2,431 unique products survived governed scope and deduplication.
- **All admitted products now have executable PDP contracts.** Product Pack 1.0.2 and the reconciled retailer catalog leave zero endpoint-ineligible products. A fully uncached pass would require 5,336 credits, or **$10.67** at $0.002 per credit.
- **Search cannot prove first-party seller status.** Seller is absent on every Search row across all ten retailers, so seller governance must use PDP evidence before matching or reporting.
- **Meijer is the only material collection-completeness concern.** 40 of 85 tasks succeeded; the other nine retailers completed at least 84 of 85 tasks.

## Context & Methods

This diagnostic audits the production Search artifacts from the Spring Valley main collection run and Kroger recovery run. The script verifies each raw object checksum, normalizes results with retailer adapters, applies the governed Vitamins & Supplements Product Pack, deduplicates at retailer + product ID grain, and plans one PDP request per admitted product using an observed positive-price context. It does **not** issue paid PDP or AI calls.

### Key Assumptions

- A positive Search price is store-level availability evidence.
- Walmart admission is limited to the supplied Spring Valley product-ID allowlist.
- Search seller fields are diagnostic only; PDP seller is authoritative for first-party governance.
- PDP freshness requires an unexpired successful snapshot for the exact governed request context.

## Data

Load the bounded audit summary and retailer-level evidence saved beside this notebook.

In [1]:
import csv
import json
from pathlib import Path

ARTIFACT_DIR = Path.cwd()
if not (ARTIFACT_DIR / "audit.json").exists():
    ARTIFACT_DIR = Path("artifacts/spring-valley-search-audit")

audit = json.loads((ARTIFACT_DIR / "audit.json").read_text(encoding="utf-8"))
with (ARTIFACT_DIR / "retailer-summary.csv").open(newline="", encoding="utf-8") as handle:
    retailers = list(csv.DictReader(handle))
print(
    f"Loaded {len(retailers)} retailer summaries and "
    f"{audit['checks']['raw_search_rows']:,} Search rows."
)

Loaded 10 retailer summaries and 20,314 Search rows.\n

## Results

### 1. Validate the audit invariants

These assertions are the minimum trust gate for using the audit to budget PDP enrichment.

In [2]:
checks = audit["checks"]
assert checks["task_count"] == checks["successful_tasks"] + checks["failed_tasks"]
assert checks["raw_artifact_checksum_failures"] == []
assert checks["pdp_ineligible_requests"] == 0
assert checks["search_admitted_products"] == checks["pdp_calls_required"]
assert (
    sum(int(row["search_admitted_products"]) for row in retailers)
    == checks["search_admitted_products"]
)
assert sum(int(row["pdp_credits_required"]) for row in retailers) == checks["pdp_credits_required"]
print("All invariants passed.")

All invariants passed.\n

### 2. Recompute headline rates and cost

The cost is a worst-case uncached estimate; the production executor should recheck the 30-day cache immediately before enqueueing.

In [3]:
task_success_rate = checks["successful_tasks"] / checks["task_count"]
admission_rate = checks["search_admitted_products"] / checks["unique_raw_products"]
positive_price_rows = sum(int(row["positive_price_rows"]) for row in retailers)
positive_price_rate = positive_price_rows / checks["raw_search_rows"]
estimated_cost_usd = checks["pdp_credits_required"] * 0.002
print(f"Task success rate: {task_success_rate:.1%}")
print(f"Unique-product admission rate: {admission_rate:.1%}")
print(f"Positive-price row rate: {positive_price_rate:.1%}")
print(
    f"Uncached PDP estimate: {checks['pdp_credits_required']:,} credits "
    f"= ${estimated_cost_usd:,.2f}"
)

Task success rate: 94.5%\nUnique-product admission rate: 28.8%\nPositive-price row rate: 95.1%\nUncached PDP estimate: 5,336 credits = $10.67\n

### 3. Identify retailer-level completeness risk

Meijer is the sole material outlier; its task success rate is below 50%, while every other retailer is at or above 98.8%.

In [4]:
retailer_success = []
for row in retailers:
    succeeded = int(row["tasks_succeeded"])
    failed = int(row["tasks_failed"])
    total = succeeded + failed
    retailer_success.append(
        (succeeded / total if total else 0, row["retailer_id"], succeeded, total)
    )
for rate, retailer_id, succeeded, total in sorted(retailer_success):
    print(f"{retailer_id:22} {rate:6.1%}  {succeeded}/{total}")

meijer_us               47.1%  40/85\namazon_us_same_day      98.8%  84/85\nwalgreens_us            98.8%  84/85\nbjs_us                 100.0%  85/85\ncostco_us              100.0%  85/85\ncvs_us                 100.0%  85/85\nkroger_us              100.0%  85/85\nsams_club_us           100.0%  85/85\ntarget_us              100.0%  85/85\nwalmart_us             100.0%  85/85\n

### 4. Verify seller and field limitations

Search contains complete identifiers, URLs, images, and price fields, but no seller evidence. Brand and sponsorship also have retailer-specific gaps that PDP enrichment must fill where available.

In [5]:
field_audit = audit["field_audit"]
seller_present = [
    retailer_id
    for retailer_id, fields in field_audit.items()
    if set(fields["raw_search_sellers"]) != {"<missing>"}
]
brand_missing = sorted(
    retailer_id
    for retailer_id, fields in field_audit.items()
    if fields["canonical_field_coverage_pct"]["brand"] == 0
)
sponsorship_missing = sorted(
    retailer_id
    for retailer_id, fields in field_audit.items()
    if fields["canonical_field_coverage_pct"]["is_sponsored"] == 0
)
print(f"Retailers with any Search seller evidence: {seller_present}")
print(f"Retailers with 0% Search brand coverage: {brand_missing}")
print(f"Retailers with 0% sponsorship coverage: {sponsorship_missing}")

Retailers with any Search seller evidence: []\nRetailers with 0% Search brand coverage: ['amazon_us_same_day', 'bjs_us', 'meijer_us', 'sams_club_us', 'walmart_us']\nRetailers with 0% sponsorship coverage: ['bjs_us', 'costco_us', 'meijer_us', 'walgreens_us']\n

## Takeaways

1. **Proceed only with a cache-aware PDP execution.** Recheck the 30-day exact-context cache at enqueue time and cap spend at the approved amount.
2. **Use PDP seller evidence as a gate before matching/reporting.** Search cannot distinguish Walmart/Target/Amazon first-party offers from marketplace sellers.
3. **Treat Meijer coverage as partial.** Matching and scorecards must carry a coverage warning until its 45 failed tasks are recovered or explicitly accepted.
4. **Retain Product Pack 1.0.2 as the governed baseline.** It removes proven pet, topical, beverage, and tea noise without excluding oral supplement language such as hair/skin/nails.
5. **Do not run AI matching until PDP normalization and first-party governance complete.** That order prevents avoidable model spend and contaminated candidates.